# Understanding UML

As an information model, CIM does not directly specify how those objects should be used in a database or data structure. Rather, it simply defines the vocabulary of how these objects should be described and what is the relationship between them. The CIM extensively uses Unified Modeling Language (UML) class diagrams to describe each class of objects, their attributes, and their relationships with other classes. UML provides 13 types of diagrams to define software architecture. One of them is the UML Class Diagram, which visually represents object hierarchies and relationships.

Class diagrams show all the attributes and associations of various classes in a particular package in a single picture. CIM classes and their attributes can be distinguished through a capitalization convention that classes and packages use **InitialCapitals**, while attributes and enumerations use **camelCase**.

A full list of all modeling rules applied to CIM are available online on the [CIM Modeling Guide](https://cim-mg.ucaiug.io/latest/section5-cim-uml-modeling-rules-and-recommendations/) website.

All diagrams within this page have been auto-generated using mermaid.js and the `cimgraph.utils` module of CIMantic Graphs for CIM17v40, which can be imported as

In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

## Classes and Objects

A **class** represents a specific type of thing that needs to be described, such as line, transformer, etc. Classes are represented as square box. **Attributes** are the properties that describe what type of thing the class represents and are shown listed inside the class.

The diagram below shows the class `ACLineSegment`, which has a set of attributes to describe the positive and zero sequence impedance of that line

In [2]:
diagram_text = utils.get_mermaid([cim.ACLineSegment])
Mermaid(diagram_text)

An **object instance** is a specific object described by that class to describe a specific piece of equipment:

In [3]:
line_object = cim.ACLineSegment(r=0.001, x=0.015, bch=0.0)
line_object.pprint() # print as json-ld

{
    "@id": "22e29426-4dba-406b-8f03-1ba849a9dfb3",
    "@type": "ACLineSegment",
    "bch": "0.0",
    "r": "0.001",
    "x": "0.015"
}


----

## Inheritance

The concept of **inheritance** allows us to define very general "parent classes" and very specific "child classes". CIM also includes the concept of *abstract* classes (which will never appear within a data file, such as `Conductor`) and *concrete* classes (which will appear in a data exchange, such as `ACLineSegment`).

In UML diagrams, lines with a triangular arrowhead indicate class inheritance. For example, in the figure below, `ACLineSegment` inherits from `Conductor`, which means that it is a specific type of conductor. 

In [4]:
diagram_text = utils.get_mermaid([cim.Conductor,cim.ACLineSegment])
Mermaid(diagram_text)

----

## Associations

**Associations** are the relationships between various objects and how they are connected to each other. In a UML diagram, associations are shown as either a connector without a terminating symbol (as shown below) or with a simple arrow to show directionality of the association. **Cardinality** refers to the multiplicity of how many objects can be associated with another.

An `ACLineSegmentPhase` object can be associated with up to one `ACLineSegment` object via the `ACLineSegmentPhase.ACLineSegment` association. In the reverse direction,an `ACLineSegment` object is associated zero to many `ACLineSegmentPhase` objects via the `ACLineSegment.ACLineSegmentPhases` association.

In [5]:
diagram_text = utils.get_mermaid([cim.ACLineSegment,cim.ACLineSegmentPhase])
Mermaid(diagram_text)

This association can also be demonstrated through object instances defined in CIM-Graph:

In [6]:
line = cim.ACLineSegment(name='new_line')
phs_a = cim.ACLineSegmentPhase(name='phase_A', ACLineSegment=line)
phs_b = cim.ACLineSegmentPhase(name='phase_B', ACLineSegment=line)
phs_c = cim.ACLineSegmentPhase(name='phase_C', ACLineSegment=line)

Per the rules of IEC 61970-552, most associations are serialized in the RDF XML instance data file in the direction of many-to-one, so an XML file for this example will show two instances of `ACLineSegmentPhase` with an assocationed with the `ACLineSegment` as shown below:

In [7]:
diagram_text = utils.get_mermaid_path(phs_a, 'ACLineSegment')
diagram_text = utils.add_mermaid_path(phs_b, 'ACLineSegment', diagram_text)
diagram_text = utils.add_mermaid_path(phs_c, 'ACLineSegment', diagram_text)
Mermaid(diagram_text)

When loading CIM models in-memory from an XML file or database, CIM-Graph will also create the reverse association to show that the `ACLineSegment` object is associated with a list of phase objects as shown below

In [8]:
line.ACLineSegmentPhases = [phs_a, phs_b, phs_c] # Association created in-memory by CIM-Graph

diagram_text = utils.get_mermaid_path(line, ['ACLineSegmentPhases','[0]'])
diagram_text = utils.add_mermaid_path(line, ['ACLineSegmentPhases','[1]'], diagram_text)
diagram_text = utils.add_mermaid_path(line, ['ACLineSegmentPhases','[2]'], diagram_text)
Mermaid(diagram_text)

----

## Aggregations

CIM uses the concept of **composition** or **aggregation** to show groupings of objects and are represented in the UML diagram as a line with diamond.For example, multiple `Substation` objects make up a `SubGeographicalRegion`, which then make up a `GeographicRegion`:

In [9]:
diagram_text = utils.get_mermaid([cim.Substation, cim.SubGeographicalRegion, cim.GeographicalRegion])
Mermaid(diagram_text)

----

## Inheritance of Class Attributes and Associations

CIM contains deep inheritance trees to allow very specific typing of the things being modeled. For example, the diagram below shows us that `ACLineSegment` is a type of `Conductor`, which is a type of `ConductingEquipment`, which is a type of `Equipment`, which is a type of `PowerSystemResource`, which is a type of `IdentifiedObject`:

In [10]:
diagram_text = utils.get_mermaid([
    cim.IdentifiedObject,
    cim.PowerSystemResource,
    cim.Equipment,
    cim.ConductingEquipment,
    cim.Conductor,
    cim.ACLineSegment])
Mermaid(diagram_text)

Objects specified using CIM inherit all the attributes and associations of their parent classes, as shown below. Due to the deep inheritance hierarchies found within CIM, a particular object will contain numerous attributes that may seem unrelated when serialized. For example, the `ACLineSegment` inherits the attributes of all its parent classes. Thus, an individual `ACLineSegment` object will contain attributes such as `mRID`, `inService`, and `length`. In most serializations, the parent classes are written out in full; however, newer approaches using knowledge graphs and JSON-LD may omit the parent class and use only the attribute name.

In [11]:
diagram_text = utils.get_mermaid([cim.ACLineSegment],show_inherited=True)
Mermaid(diagram_text)

Likewise, CIM classes also inherit all of the associations of their parent classes to other classes. As shown below, the class `ACLineSegment` inherits the associations of its parent classes to `PowerSystemResouce.Location`, `Equipment.EquipmentContainer`, and `ConductingVoltage.BaseVoltage`. Thus, when serialized, the associations of its parent classes will appear along with its inherited attributes

In [12]:
diagram_text = utils.get_mermaid([cim.ACLineSegment, cim.Conductor, cim.ConductingEquipment, cim.Equipment], show_attributes=False)
diagram_text = utils.add_mermaid_path(cim.ConductingEquipment, 'BaseVoltage', diagram_text)
diagram_text = utils.add_mermaid_path(cim.Equipment, 'EquipmentContainer', diagram_text)
Mermaid(diagram_text)

----